# 🌾 Project 4: Prediction of Agriculture Crop Production in India
**Internship Project | UpSkill Campus & UCT**

| Detail | Info |
|---|---|
| **Author** | Suryakant Prajapati |
| **Internship** | Free Summer Internship – Data Science & ML |
| **Company** | UpSkill Campus & UCT |
| **Domain** | Agriculture / Machine Learning |
| **Tools** | Python, Pandas, Scikit-learn, Matplotlib, Seaborn |

---

## 📌 About UCT
UCT (Universal Career Transformer) is an ed-tech company focused on upskilling students with industry-relevant knowledge through internships and hands-on projects in Data Science, Machine Learning, and AI.

## 📌 Problem Statement
India has over 1.3 billion people and agriculture is the backbone of its economy. Given historical data (2001–2014) of crop cultivation across Indian states, this project predicts **crop production (in tons)** using features like crop type, state, season, and area cultivated.

## 📌 Objective
Build and compare ML models to accurately predict agricultural crop production across different Indian states and seasons.

## ✅ STEP 1: Install & Import Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('✅ All libraries imported successfully!')

## ✅ STEP 2: Load Dataset

> **Note for Kaggle users:** If loading via Kaggle API, store credentials using environment variables — never hardcode your API key.
> ```python
> import os
> os.environ['KAGGLE_USERNAME'] = 'your_username'   # Replace before running
> os.environ['KAGGLE_KEY'] = 'your_api_key'         # Replace before running
> ```
> **Never upload your actual API key to GitHub.**

In [ ]:
# ✅ SAFE METHOD - Load directly from public GitHub (no API key needed)
url = 'https://raw.githubusercontent.com/loki4514/Crop-Production-Statistics---India/main/crop_production.csv'

try:
    df = pd.read_csv(url)
    print('✅ Dataset loaded from GitHub!')
except:
    # Fallback: Upload manually
    print('GitHub URL failed. Please upload the CSV manually:')
    print('1. Download from: https://www.kaggle.com/datasets/abhinand05/crop-production-in-india')
    print('2. Upload to Colab using the file icon on the left sidebar')
    print('3. Run: df = pd.read_csv("crop_production.csv")')

print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head(10)

## ✅ STEP 3: Explore the Data (EDA)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Statistics ===')
df.describe()

In [ ]:
print(f'Unique States : {df["State_Name"].nunique()}')
print(f'Unique Crops  : {df["Crop"].nunique()}')
print(f'Unique Seasons: {df["Season"].nunique()}')
print(f'Year Range    : {df["Crop_Year"].min()} - {df["Crop_Year"].max()}')

## ✅ STEP 4: Data Visualization

In [ ]:
# Chart 1: Top 10 Crops
plt.figure(figsize=(12, 5))
top_crops = df.groupby('Crop')['Production'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_crops.values, y=top_crops.index, palette='viridis')
plt.title('🌾 Top 10 Crops by Total Production in India', fontsize=14)
plt.xlabel('Total Production')
plt.tight_layout()
plt.savefig('chart1_top_crops.png', dpi=150, bbox_inches='tight')  # Save for GitHub
plt.show()
print('✅ Saved: chart1_top_crops.png')

In [ ]:
# Chart 2: Top 10 States
plt.figure(figsize=(12, 5))
top_states = df.groupby('State_Name')['Production'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_states.values, y=top_states.index, palette='magma')
plt.title('🏛️ Top 10 States by Crop Production', fontsize=14)
plt.xlabel('Total Production')
plt.tight_layout()
plt.savefig('chart2_top_states.png', dpi=150, bbox_inches='tight')  # Save for GitHub
plt.show()
print('✅ Saved: chart2_top_states.png')

In [ ]:
# Chart 3: Production by Season
plt.figure(figsize=(10, 5))
season_prod = df.groupby('Season')['Production'].sum().sort_values(ascending=False)
sns.barplot(x=season_prod.index, y=season_prod.values, palette='coolwarm')
plt.title('📅 Crop Production by Season', fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart3_season.png', dpi=150, bbox_inches='tight')  # Save for GitHub
plt.show()
print('✅ Saved: chart3_season.png')

In [ ]:
# Chart 4: Year-wise Production Trend
plt.figure(figsize=(12, 5))
year_prod = df.groupby('Crop_Year')['Production'].sum()
plt.plot(year_prod.index, year_prod.values, marker='o', color='green', linewidth=2)
plt.title('📈 Year-wise Crop Production Trend', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Total Production')
plt.grid(True)
plt.tight_layout()
plt.savefig('chart4_year_trend.png', dpi=150, bbox_inches='tight')  # Save for GitHub
plt.show()
print('✅ Saved: chart4_year_trend.png')

## ✅ STEP 5: Data Preprocessing

In [ ]:
df = df.dropna(subset=['Production', 'Area'])
upper_limit = df['Production'].quantile(0.99)
df = df[df['Production'] <= upper_limit]

le_state    = LabelEncoder()
le_crop     = LabelEncoder()
le_season   = LabelEncoder()
le_district = LabelEncoder()

df['State_Encoded']    = le_state.fit_transform(df['State_Name'])
df['Crop_Encoded']     = le_crop.fit_transform(df['Crop'])
df['Season_Encoded']   = le_season.fit_transform(df['Season'].str.strip())
df['District_Encoded'] = le_district.fit_transform(df['District_Name'])

print(f'✅ Data cleaned. Final shape: {df.shape}')

## ✅ STEP 6: Train ML Models

In [ ]:
X = df[['State_Encoded','District_Encoded','Crop_Year','Season_Encoded','Crop_Encoded','Area']]
y = df['Production']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print('✅ Both models trained!')

## ✅ STEP 7: Results & Evaluation

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE':   [mean_absolute_error(y_test, lr_pred), mean_absolute_error(y_test, rf_pred)],
    'RMSE':  [np.sqrt(mean_squared_error(y_test, lr_pred)), np.sqrt(mean_squared_error(y_test, rf_pred))],
    'R² Score': [r2_score(y_test, lr_pred), r2_score(y_test, rf_pred)]
})
print(results.to_string(index=False))
print('\n✅ Random Forest performs better (higher R², lower MAE)')

In [ ]:
# Chart 5: Actual vs Predicted
plt.figure(figsize=(10, 5))
plt.scatter(y_test[:300], rf_pred[:300], alpha=0.5, color='blue', label='Predicted')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect')
plt.xlabel('Actual Production')
plt.ylabel('Predicted Production')
plt.title('🎯 Actual vs Predicted (Random Forest)', fontsize=13)
plt.legend()
plt.tight_layout()
plt.savefig('chart5_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart5_actual_vs_predicted.png')

In [ ]:
# Chart 6: Feature Importance
importances = rf.feature_importances_
feature_names = X.columns.tolist()

plt.figure(figsize=(8, 5))
sns.barplot(x=importances, y=feature_names, palette='Blues_r')
plt.title('🔍 Feature Importance (Random Forest)', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('chart6_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart6_feature_importance.png')

## ✅ STEP 8: Download All Charts for GitHub Upload

In [ ]:
# Download all saved chart images from Colab to your computer
from google.colab import files

chart_files = [
    'chart1_top_crops.png',
    'chart2_top_states.png',
    'chart3_season.png',
    'chart4_year_trend.png',
    'chart5_actual_vs_predicted.png',
    'chart6_feature_importance.png'
]

for f in chart_files:
    files.download(f)
    print(f'⬇️ Downloaded: {f}')

print('\n✅ All charts downloaded! Upload these to your GitHub images/ folder.')

## 📋 Summary

### Results
- **Best Model:** Random Forest Regressor
- **Key Finding:** Area cultivated is the most important predictor of crop production
- **States:** Uttar Pradesh and Madhya Pradesh lead in total crop production

### My Learnings
1. Loaded and explored real-world Indian agricultural data
2. Applied Label Encoding to convert categorical text into numbers
3. Trained Linear Regression and Random Forest models
4. Evaluated models using MAE, RMSE, and R² Score
5. Visualized feature importance to understand key drivers

### Future Improvements
- Add weather/rainfall data for better accuracy
- Try XGBoost or Gradient Boosting
- Deploy as a Streamlit web app

---
**Submitted by:** Suryakant Prajapati | UpSkill Campus Internship